In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class PositionalEmbedding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()

        self.pos_embedding = nn.Embedding(max_len, d_model)

    def forward(self, idx):  # B,T
        B, T = idx.shape
        positions = torch.arange(T, device=idx.device)  # T
        positions = positions.unsqueeze(0).expand(B, T)  # B,T
        return self.pos_embedding(positions)


class GenreEmbedding(nn.Module):
    def __init__(self, num_genres, d_model):
        super().__init__()

        self.embedding = nn.Embedding(
            num_genres,
            d_model,
        )

    def forward(self, genres):  # B, T, G (multi-hot: 0/1)
        # genres: binary indicators

        # B,T,G -> B,T,G,d
        emb = self.embedding.weight  # G,d
        emb = emb.unsqueeze(0).unsqueeze(0)  # 1,1,G,d

        genres = genres.unsqueeze(-1)  # B,T,G,1

        genres_emb = emb * genres  # mask active genres
        genres_emb = genres_emb.sum(dim=2)  # B,T,d

        return genres_emb


class BERT4RecEmbedding(nn.Module):
    def __init__(self, d_model, max_len, vocab_size, dropout=0.1):
        super().__init__()

        self.tok_embedding = nn.Embedding(
            vocab_size,  # include pad &
            d_model,
            padding_idx=0,  # CRITICAL
        )

        self.pos_embedding = nn.Embedding(max_len, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, idx):
        B, T = idx.shape

        positions = torch.arange(T, device=idx.device)
        positions = positions.unsqueeze(0).expand(B, T)

        tok_emb = self.tok_embedding(idx)
        pos_emb = self.pos_embedding(positions)

        emb = tok_emb + pos_emb
        emb = self.dropout(emb)

        return emb


class MetaBERT4RecEmbedding(nn.Module):
    def __init__(self, d_model, max_len, vocab_size, num_genres, dropout=0.1):
        super().__init__()

        self.tok_embedding = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=0,  # CRITICAL
        )

        self.pos_embedding = nn.Embedding(max_len, d_model)

        self.genre_embedding = GenreEmbedding(num_genres, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, idx, genres):
        B, T = idx.shape

        positions = torch.arange(T, device=idx.device)
        positions = positions.unsqueeze(0).expand(B, T)

        tok_emb = self.tok_embedding(idx)  # B,T,d
        pos_emb = self.pos_embedding(positions)
        genre_emb = self.genre_embedding(genres)

        emb = tok_emb + pos_emb + genre_emb
        emb = self.dropout(emb)

        return emb


class FFN(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        self.gelu = nn.GELU()
        self.l1 = nn.Linear(d_model, d_model * 4)
        self.l2 = nn.Linear(d_model * 4, d_model)

    def forward(self, x):
        return self.l2(self.gelu(self.l1(x)))


class PFFN(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        self.ffn = FFN(d_model)

    def forward(self, x):
        return self.ffn(x)


class Trm(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()

        self.mh = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True
        )
        self.pffn = PFFN(d_model)
        self.dropout = nn.Dropout(p=dropout)
        self.layer_norm = nn.LayerNorm(normalized_shape=d_model)

    def forward(self, x, key_padding_mask=None):
        attn_out, _ = self.mh(
            x,
            x,
            x,
            key_padding_mask=key_padding_mask,
        )
        x = x + self.dropout(attn_out)
        x = self.layer_norm(x)

        pffn_out = self.pffn(x)
        x = x + self.dropout(pffn_out)
        x = self.layer_norm(x)

        return x


class BERT4Rec(nn.Module):
    def __init__(
        self, max_len, d_model, n_heads, n_layers, vocab_size, dropout=0.1
    ):
        super().__init__()

        self.embedding = BERT4RecEmbedding(
            d_model, max_len, vocab_size, dropout=dropout
        )
        self.trm_layers = nn.ModuleList(
            [Trm(d_model, n_heads, dropout=dropout) for _ in range(n_layers)]
        )
        self.proj = nn.Linear(d_model, d_model)  # a = Wa + b
        self.gelu = nn.GELU()
        self.output_bias = nn.Parameter(torch.zeros(vocab_size))

    def forward(
        self,
        idx,
        key_padding_mask,
        candidates=None,
    ):
        B, _ = idx.shape
        h = self.embedding(idx)
        for layer in self.trm_layers:
            h = layer(h, key_padding_mask=key_padding_mask)

        if candidates is not None:
            h_last = h[:, -1, :]
            z = self.gelu(self.proj(h_last))
            candidates_embedding = self.embedding.tok_embedding(candidates)
            logits = torch.matmul(
                z.unsqueeze(1), candidates_embedding.transpose(1, 2)
            ).squeeze(1)
            logits = logits + self.output_bias[candidates]
        else:
            z = self.gelu(self.proj(h))
            logits = torch.matmul(z, self.embedding.tok_embedding.weight.T)
            logits = logits + self.output_bias

        return logits


class MetaBERT4Rec(nn.Module):
    def __init__(
        self,
        max_len,
        num_genres,
        d_model,
        n_heads,
        n_layers,
        vocab_size,
        dropout=0.1,
    ):
        super().__init__()

        self.embedding = MetaBERT4RecEmbedding(
            d_model=d_model,
            max_len=max_len,
            vocab_size=vocab_size,
            num_genres=num_genres,
            dropout=dropout,
        )
        self.trm_layers = nn.ModuleList(
            [Trm(d_model, n_heads, dropout=dropout) for _ in range(n_layers)]
        )
        self.proj = nn.Linear(d_model, d_model)  # a = Wa + b
        self.gelu = nn.GELU()
        self.output_bias = nn.Parameter(torch.zeros(vocab_size))

    def forward(
        self,
        idx,
        genres,
        key_padding_mask,
        candidates=None,
    ):
        B, _ = idx.shape

        h = self.embedding(idx, genres)
        for layer in self.trm_layers:
            h = layer(h, key_padding_mask=key_padding_mask)

        if candidates is not None:
            h_last = h[:, -1, :]
            z = self.gelu(self.proj(h_last))
            candidates_embedding = self.embedding.tok_embedding(candidates)
            logits = torch.matmul(
                z.unsqueeze(1), candidates_embedding.transpose(1, 2)
            ).squeeze(1)
            logits = logits + self.output_bias[candidates]
        else:
            z = self.gelu(self.proj(h))
            logits = torch.matmul(z, self.embedding.tok_embedding.weight.T)
            logits = logits + self.output_bias

        return logits


# if __name__ == "__main__":
#     from torch.utils.data import DataLoader
#     from tqdm import tqdm

#     ds = MovieLenDataset(
#         movies=movies,
#         ratings=ratings,
#         max_len=max_len,
#         min_len=min_len,
#         split="train",
#     )

#     loader = DataLoader(
#         dataset=ds,
#         batch_size=4,
#         shuffle=True,
#         num_workers=2,
#     )

#     b = next(iter(loader))

#     model = MetaBERT4Rec(
#         max_len=200,
#         d_model=64,
#         n_heads=4,
#         n_layers=6,
#         num_genres=18,
#         vocab_size=27279,
#     )

#     model.to("cuda")

#     out = model.forward(
#         idx=b["input"],
#         genres=b["genres"],
#         token_mask=b["token_mask"],
#         key_padding_mask=b["key_padding_mask"],
#         candidates=b["candidates"],
#     )

#     out.shape

In [2]:
import os
import subprocess
from zipfile import ZipFile
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from tqdm import tqdm

WEEK_IN_SEC = 604800
DAY_IN_SEC = 86400

GENRES = [
    "Action",
    "Adventure",
    "Animation",
    "Children's",
    "Comedy",
    "Crime",
    "Documentary",
    "Drama",
    "Fantasy",
    "Film-Noir",
    "Horror",
    "Musical",
    "Mystery",
    "Romance",
    "Sci-Fi",
    "Thriller",
    "War",
    "Western",
]


def get_genre_matrix(movies_df):
    """Vectorized genre encoding using Pandas dummies"""
    dummies = movies_df["genres"].str.get_dummies(sep="|")
    return dummies.reindex(columns=GENRES, fill_value=0).values


def generate_mask(seq, mask_rate):
    """
    Randomly generate a mask for the given sequence. The mask rate specify how much of the sequence is masked
    True value indicate the position will be masked.
    """
    return torch.rand(len(seq)) < mask_rate


def parse_week(ratings):
    """
    Parse the week where the current rating is on.
    ratings where the timestamp is less than 1 day away from the start of a week will be parsed as previous week
    """
    return np.where(
        (ratings["timestamp"] % WEEK_IN_SEC) > DAY_IN_SEC,
        ratings["timestamp"] // WEEK_IN_SEC,
        (ratings["timestamp"] // WEEK_IN_SEC) - 1,
    )


class MovieLenDataset(Dataset):
    """
    Args:
        movies: the movies dataframe
        ratings: the ratings dataframe
        negative_rule: the rule used to determine how negative items are sampled (popularity|trending|random)
        top_k: the k movies will be used for negative sample
        min_len: the minimum user history length to be used, otherwise that user will be removed.
        max_len: the maximum user history length to be used, otherwise that user will be removed.
        mask_rate: the proportion of the sequence to be masked randomly
        split: the target split the dataset is used for (train|val|test)
    """

    def __init__(
        self,
        movies,
        ratings,
        min_len=5,
        max_len=200,
        negative_rule="popularity",
        strides=1,
        mask_rate=0.2,
        top_k=100,
        split="train",
    ):
        super().__init__()

        self.split = split
        self.negative_rule = negative_rule
        self.max_len = max_len
        self.mask_rate = mask_rate
        self.top_k = top_k
        self.negative_samples = []

        self._prepare(movies, ratings)
        self._build_sequences(min_len, strides)
        self.MASK_ID = len(self.movies) + 1

        if self.split == "train":
            return

        if self.negative_rule == "popularity":
            movies_by_popularity = (
                self.ratings["movie_idx"].value_counts().index
            )
            for i in tqdm(range(len(self.seqs))):
                seq = self.seqs[i]["seq"]
                sample = movies_by_popularity[~movies_by_popularity.isin(seq)][
                    : self.top_k
                ].to_list()
                self.negative_samples.append(sample)
        elif self.negative_rule == "trending":
            movies_by_trending = (
                self.ratings.groupby(["movie_idx", "week"])["movieId"]
                .agg("count")
                .to_frame("count")
                .reset_index()
                .sort_values(["week", "count"], ascending=False)
            )

            for i in tqdm(range(len(self.seqs))):
                seq = self.seqs[i]["seq"]
                week = self.seqs[i]["week"]
                sample = (
                    movies_by_trending[movies_by_trending["week"] == week]
                    .head(self.top_k)["movie_idx"]
                    .to_list()
                )
                self.negative_samples.append(sample)
        elif self.negative_rule == "random":
            for i in tqdm(range(len(self.seqs))):
                seq = self.seqs[i]["seq"]
                sample = (
                    self.movies[~self.movies["movie_idx"].isin(seq)][
                        "movie_idx"
                    ]
                    .sample(self.top_k)
                    .to_list()
                )
                self.negative_samples.append(sample)

    def _prepare(self, movies, ratings):
        ratings["week"] = parse_week(ratings)
        id2idx = {id: idx + 1 for idx, id in enumerate(movies["movieId"])}
        ratings["movie_idx"] = ratings["movieId"].map(id2idx)
        movies["movie_idx"] = movies["movieId"].map(id2idx)
        self.genres_lookup = np.vstack(
            [np.zeros(len(GENRES)), get_genre_matrix(movies)]
        )
        self.movies = movies
        self.ratings = ratings

    def _build_sequences(self, min_len, strides):
        grouped = self.ratings.sort_values("timestamp").groupby("userId")
        user_data = grouped.agg({"movie_idx": list, "week": list})

        iterator = tqdm(
            user_data.iterrows(),
            total=len(user_data),
            desc=f"Initialize dataset for {self.split}",
        )

        seqs = []
        for _, row in iterator:
            hist, weeks = row["movie_idx"], row["week"]
            if len(hist) < min_len:
                continue

            if self.split == "train":
                for i in range(
                    0, max(len(hist) - self.max_len - 2, 1), strides
                ):
                    seq = hist[i : i + self.max_len]
                    seqs.append({"seq": seq})

            elif self.split == "val" or self.split == "test":
                offset = 1 if self.split == "val" else 0
                idx_end = len(hist) - offset
                seq = hist[max(idx_end - self.max_len, 0) : idx_end]
                target_week = weeks[-1]
                seqs.append({"seq": seq, "week": target_week})

        self.seqs = seqs

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        seq = self.seqs[idx]["seq"]
        genres = self.genres_lookup[seq]
        seq = torch.tensor(seq)
        genres = torch.from_numpy(genres).long()
        pad = (max(0, self.max_len - len(seq)), 0)
        padded_seq = F.pad(seq, pad, value=0)
        padded_genres = F.pad(genres, (0, 0, pad[0], pad[1]))
        key_padding_mask = padded_seq == 0

        if self.split == "train":
            token_mask = generate_mask(seq, self.mask_rate)
            padded_token_mask = F.pad(token_mask, pad, value=False)
            label = padded_seq.clone()
            padded_seq[padded_token_mask] = self.MASK_ID

            return {
                "input": padded_seq,
                "label": label,
                "genres": padded_genres,
                "token_mask": padded_token_mask,
                "key_padding_mask": key_padding_mask,
            }
        elif self.split == "val" or self.split == "test":
            negatives = torch.tensor(self.negative_samples[idx])
            negatives_pad = (max(0, self.top_k - len(negatives)), 0)
            padded_negatives = F.pad(negatives, negatives_pad)
            token_mask = torch.tensor([False] * (len(seq) - 1) + [True])
            padded_token_mask = F.pad(token_mask, pad, value=False)
            label = padded_seq.clone()
            padded_seq[padded_token_mask] = self.MASK_ID
            target = seq[-1]

            return {
                "input": padded_seq,
                "label": label,
                "genres": padded_genres,
                "token_mask": padded_token_mask,
                "key_padding_mask": key_padding_mask,
                "candidates": torch.cat(
                    (padded_negatives, target.unsqueeze(0))
                ),
            }


# if __name__ == "__main__":
#     ds_url = "https://files.grouplens.org/datasets/movielens/ml-20m.zip"
#     temp_dir = "/tmp"

#     subprocess.run(["wget", "-P", temp_dir, ds_url])

#     with ZipFile(os.path.join(temp_dir, "ml-20m.zip")) as z_obj:
#         z_obj.extractall(path=temp_dir)

#     movies_path = os.path.join(temp_dir, "ml-20m", "movies.csv")
#     ratings_path = os.path.join(temp_dir, "ml-20m", "ratings.csv")
#     tags_path = os.path.join(temp_dir, "ml-20m", "tags.csv")
#     links_path = os.path.join(temp_dir, "ml-20m", "links.csv")
#     genome_tags_path = os.path.join(temp_dir, "ml-20m", "genome-tags.csv")
#     genome_scores_path = os.path.join(temp_dir, "ml-20m", "genome-scores.csv")

#     movies = pd.read_csv(movies_path)
#     ratings = pd.read_csv(ratings_path)
#     tags = pd.read_csv(tags_path)
#     links = pd.read_csv(links_path)
#     genome_tags = pd.read_csv(genome_tags_path)
#     genome_scores = pd.read_csv(genome_scores_path)

#     dss = {}

#     dss["train"] = MovieLenDataset(
#         movies=movies,
#         ratings=ratings,
#         max_len=200,
#         split="train",
#     )

#     s = dss["train"][2]
#     print(s["input"].shape)
#     print(s["input"])
#     print(s["token_mask"].shape)
#     print(s["token_mask"])
#     print(s["key_padding_mask"].shape)
#     print(s["key_padding_mask"])
#     print(s["genres"].shape)
#     print(s["genres"])

#     s["input"].to("cuda")

#     for rule in ["trending"]:
#         print("==========================================")
#         dss[rule] = MovieLenDataset(
#             movies=movies,
#             ratings=ratings,
#             max_len=200,
#             split="test",
#             negative_rule=rule,
#         )
#         s = dss[rule][0]
#         print(s["input"].shape)
#         print(s["input"])
#         print(s["target"].shape)
#         print(s["target"])
#         print(s["token_mask"].shape)
#         print(s["token_mask"])
#         print(s["key_padding_mask"].shape)
#         print(s["key_padding_mask"])
#         print(s["genres"].shape)
#         print(s["genres"])
#         print(s["candidates"].shape)
#         print(s["candidates"])

In [ ]:
import pandas as pd
import os
import subprocess
from zipfile import ZipFile
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm
import time

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

# ==  Variables == #

batch_size = 64
num_epochs = 50
val_iter = 5
mask_rate = 0.2
max_len = 200
min_len = 5
d_model = 32
n_heads = 2
n_layers = 2
dropout = 0.2
lr = 1e-5
top_k = 200

model_name = "metabert4rec"

base_dir = ""
experiment_dir = f"{model_name}_{d_model}"
if not os.path.isdir(os.path.join(base_dir, experiment_dir)):
    os.mkdir(os.path.join(base_dir, experiment_dir))

checkpoint_path = os.path.join(base_dir, experiment_dir, "checkpoint.pt")
losses_path = os.path.join(base_dir, experiment_dir, "losses.csv")
validation_path = os.path.join(base_dir, experiment_dir, "validation.csv")

ds_url = "https://files.grouplens.org/datasets/movielens/ml-32m.zip"
temp_dir = "/tmp"

cuda


In [ ]:
movies_path = os.path.join(temp_dir, "ml-32m", "movies.csv")
ratings_path = os.path.join(temp_dir, "ml-32m", "ratings.csv")

movies = pd.read_csv(movies_path)
ratings = pd.read_csv(ratings_path)

# == Initialize datasets == #

train_ds = MovieLenDataset(
    movies=movies,
    ratings=ratings,
    max_len=max_len,
    min_len=min_len,
    strides=20,
    split="train",
)

popularity_val_ds = MovieLenDataset(
    movies=movies,
    ratings=ratings,
    max_len=max_len,
    min_len=min_len,
    split="val",
    top_k=top_k,
    negative_rule="popularity",
)

random_val_ds = MovieLenDataset(
    movies=movies,
    ratings=ratings,
    max_len=max_len,
    min_len=min_len,
    split="val",
    top_k=top_k,
    negative_rule="random",
)

trending_val_ds = MovieLenDataset(
    movies=movies,
    ratings=ratings,
    max_len=max_len,
    min_len=min_len,
    split="val",
    top_k=top_k,
    negative_rule="trending",
)

train_loader = DataLoader(
    dataset=train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)

popularity_val_loader = DataLoader(
    dataset=popularity_val_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
)

random_val_loader = DataLoader(
    dataset=random_val_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
)

trending_val_loader = DataLoader(
    dataset=trending_val_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
)

# == Load checkpoint == #

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path)
else:
    checkpoint = None

# == Model == #

model = MetaBERT4Rec(
    max_len=max_len,
    d_model=d_model,
    n_heads=n_heads,
    n_layers=n_layers,
    num_genres=18,
    vocab_size=len(movies) + 2,
)


def init_weights(module):
    if isinstance(module, (nn.Linear, nn.Embedding)):
        if not module.weight.requires_grad:
            return

        nn.init.trunc_normal_(module.weight, std=0.02)
        if hasattr(module, "bias") and module.bias is not None:
            nn.init.zeros_(module.bias)


model.apply(init_weights)

if checkpoint is not None:
    model.load_state_dict(checkpoint["model"])

model.to(device)


# == Training tools == #

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    params=model.parameters(),
    lr=lr,
)
scheduler = CosineAnnealingLR(
    optimizer=optimizer,
    T_max=num_epochs,
)

if checkpoint is not None:
    optimizer.load_state_dict(checkpoint["optimizer"])
    scheduler.load_state_dict(checkpoint["scheduler"])

# == losses and validation dataframe == #

if os.path.exists(losses_path):
    losses_df = pd.read_csv(losses_path)
else:
    losses_df = pd.DataFrame(
        columns=[
            "epoch",
            "step",
            "loss",
        ]
    )

if os.path.exists(validation_path):
    validation_df = pd.read_csv(validation_path)
else:
    columns = [
        "epoch",
        "Recall@1",
        "Recall@5",
        "Recall@10",
        "MRR@1",
        "MRR@5",
        "MRR@10",
        "MRR",
        "NDCG@1",
        "NDCG@5",
        "NDCG@10",
        "MeanRank",
    ]
    validation_df = pd.DataFrame(columns=columns)

# == Training script == #


def validate_one_epoch(
    model,
    val_loader,
    device,
    validation_df,
    val_type,
    epoch,
    K_list=[1, 5, 10],
):
    model.eval()

    # Accumulators
    metrics = {
        f"{metric}@{k}": 0.0
        for metric in ["Recall", "NDCG", "MRR"]
        for k in K_list
    }

    # Global metrics
    metrics["MRR"] = 0.0
    metrics["MeanRank"] = 0.0

    total_samples = 0
    st = time.perf_counter()

    with torch.no_grad():
        for batch in tqdm(val_loader):
            idx = batch["input"].to(device)
            genres = batch["genres"].to(device)
            key_padding_mask = batch["key_padding_mask"].to(device)
            candidates = batch["candidates"].to(device)  # [B, C]

            # Forward
            logits = model.forward(
                idx=idx,
                genres=genres,
                key_padding_mask=key_padding_mask,
                candidates=candidates,
            )  # [B, C]

            B, C = logits.shape
            target_idx = C - 1  # always last position

            # Sort logits
            sorted_indices = torch.argsort(logits, dim=1, descending=True)

            # Find rank of target
            target_positions = (sorted_indices == target_idx).nonzero(
                as_tuple=False
            )

            ranks = torch.zeros(B, device=device, dtype=torch.long)
            ranks[target_positions[:, 0]] = (
                target_positions[:, 1] + 1
            )  # 1-indexed

            ranks_float = ranks.float()

            # === Metrics ===
            for K in K_list:
                hit = (ranks <= K).float()

                # Recall@K
                metrics[f"Recall@{K}"] += hit.sum().item()

                # NDCG@K
                ndcg = torch.where(
                    hit > 0,
                    1.0 / torch.log2(ranks_float + 1),
                    torch.zeros_like(hit),
                )
                metrics[f"NDCG@{K}"] += ndcg.sum().item()

                # MRR@K
                mrr_k = torch.where(
                    ranks <= K,
                    1.0 / ranks_float,
                    torch.zeros_like(ranks_float),
                )
                metrics[f"MRR@{K}"] += mrr_k.sum().item()

            # === Global MRR ===
            metrics["MRR"] += (1.0 / ranks_float).sum().item()

            # === Mean Rank ===
            metrics["MeanRank"] += ranks_float.sum().item()

            total_samples += B

    # Average
    for key in metrics:
        metrics[key] /= total_samples

    et = time.perf_counter()
    total_run_time = et - st

    # Append
    row = {
        "epoch": epoch,
        "val_type": val_type,
        "sec_per_batch": total_run_time / total_samples,
        **metrics,
    }
    validation_df.loc[len(validation_df)] = row

    return validation_df


def train_one_epoch(model, optimizer, batch):
    model.train()

    idx = batch["input"].to(device)
    label = batch["label"].to(device)
    genres = batch["genres"].to(device)
    token_mask = batch["token_mask"].to(device)
    key_padding_mask = batch["key_padding_mask"].to(device)

    logits = model.forward(
        idx=idx,
        key_padding_mask=key_padding_mask,
        genres=genres,
    )

    flatten_token_mask = torch.flatten(token_mask)
    V = logits.shape[2]
    y_pred = logits.view(-1, V)[flatten_token_mask]
    y_true = torch.flatten(label)[flatten_token_mask]

    loss = criterion(y_pred, y_true)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()


start_epoch = 1 if checkpoint is None else checkpoint["epoch"] + 1
for epoch in range(start_epoch, num_epochs + 1):
    pbar = tqdm(enumerate(train_loader), total=len(train_loader))
    for step, batch in pbar:
        loss = train_one_epoch(model, optimizer, batch)
        losses_df.loc[len(losses_df)] = {
            "epoch": epoch,
            "step": step,
            "loss": loss,
        }

        pbar.set_description(desc=f"Loss: {loss}")

    scheduler.step()

    epoch_loss = losses_df[losses_df["epoch"] == epoch]["loss"].mean()
    print(f"{epoch}/{num_epochs}: Average loss: {epoch_loss}")

    if epoch % val_iter == 0:
        validation_df = validate_one_epoch(
            model=model,
            val_loader=popularity_val_loader,
            val_type="popularity",
            device=device,
            validation_df=validation_df,
            epoch=epoch,
        )
        validation_df = validate_one_epoch(
            model=model,
            val_loader=random_val_loader,
            val_type="random",
            device=device,
            validation_df=validation_df,
            epoch=epoch,
        )
        validation_df = validate_one_epoch(
            model=model,
            val_loader=trending_val_loader,
            val_type="trending",
            device=device,
            validation_df=validation_df,
            epoch=epoch,
        )
        validation_df.to_csv(validation_path)

        print("Validation result")
        print(validation_df[validation_df["epoch"] == epoch])

    torch.save(
        {
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
        },
        checkpoint_path,
    )
    losses_df.to_csv(losses_path)

Loss: 8.357605934143066: 100%|██████████| 7464/7464 [05:47<00:00, 21.49it/s] 


1/50: Average loss: 9.013001832420292


Loss: 8.036578178405762: 100%|██████████| 7464/7464 [05:53<00:00, 21.10it/s] 


2/50: Average loss: 8.205900147924597


Loss: 7.496179580688477: 100%|██████████| 7464/7464 [06:00<00:00, 20.70it/s] 


...

49/50: Average loss: 3.268166815384919


Loss: 3.4471704959869385: 100%|██████████| 7464/7464 [07:26<00:00, 16.70it/s]


50/50: Average loss: 3.2683340143352457


100%|██████████| 2164/2164 [00:24<00:00, 88.00it/s]


Validation result
    epoch  Recall@1  Recall@5  Recall@10     MRR@1     MRR@5    MRR@10  \
27     50  0.501534  0.821854   0.957348  0.501534  0.622162  0.640153   
28     50  0.886976  0.988281   0.995581  0.886976  0.930783  0.931823   
29     50  0.209050  0.775238   0.929289  0.209050  0.437194  0.458231   

         MRR    NDCG@1    NDCG@5   NDCG@10  MeanRank  
27  0.643609  0.501534  0.671993  0.715712  3.115089  
28  0.932031  0.886976  0.945443  0.947870  1.419732  
29  0.463209  0.209050  0.522365  0.572662  4.073939  
